In [1]:
# =========================================================
# 📦 IMPORTS + SETUP
# =========================================================
import sys, os, random, time, gc
from pathlib import Path
import yaml
import torch
import pandas as pd

# ---------------------------------------------------------
# 🔥 PROJECT ROOT SETUP
# ---------------------------------------------------------
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] 
     if (p / "src").exists() and (p / "configs").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# 🔥 REGISTER CUSTOM MODULES
# ---------------------------------------------------------
import src
import ultralytics.nn.tasks as _tasks
import ultralytics.nn.modules as _modules

from src.custom_modules import M_C3k2, WeightedConcat, HybridSPDConv_3
from src.spd_conv import SPDConv, SPDHybrid, SPDHybrid_NO_Fuse

for name, cls in {
    "M_C3k2": M_C3k2,
    "WeightedConcat": WeightedConcat,
    "HybridSPDConv_3": HybridSPDConv_3,
    "SPDConv": SPDConv,
    "SPDHybrid": SPDHybrid,
    "SPDHybrid_NO_Fuse": SPDHybrid_NO_Fuse,
}.items():
    _tasks.__dict__[name] = cls
    _modules.__dict__[name] = cls

from ultralytics import YOLO


# =========================================================
# 🔁 SEED CONTROL
# =========================================================
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# 📂 LOAD CONFIG
# =========================================================
def load_config():
    cfg_path = PROJECT_ROOT / "configs" / "base.yaml"
    with open(cfg_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg["model"] = str((PROJECT_ROOT / cfg["model"]).resolve())
    cfg["experiment"]["project_dir"] = str(
        (PROJECT_ROOT / cfg["experiment"]["project_dir"]).resolve()
    )
    cfg["data"]["config"] = str(
        (PROJECT_ROOT / cfg["data"]["config"]).resolve()
    )

    return cfg


# =========================================================
# 🚀 MAIN PIPELINE
# =========================================================
def run():
    cfg = load_config()

    model_name = cfg["model"]
    training = cfg["training"]
    experiment = cfg["experiment"]
    data_cfg = cfg["data"]["config"]

    seeds = experiment["seeds"]

    all_results = []

    print("\n🚀 Starting Experiments...\n")

    # =====================================
    # 🔁 LOOP OVER SEEDS
    # =====================================
    for seed in seeds:

        print(f"\n========== SEED {seed} ==========\n")

        set_seed(seed)

        exp_name = f"{experiment['name']}_seed{seed}"

        model = YOLO(model_name)

        # --------------------------
        # TRAIN
        # --------------------------
        start = time.time()

        results = model.train(
            data=data_cfg,
            imgsz=training["imgsz"],
            batch=training["batch"],
            epochs=training["epochs"],
            optimizer=training["optimizer"],
            lr0=training["lr0"],
            workers=training["workers"],
            seed=seed,
            project=experiment["project_dir"],
            name=exp_name,
        )

        train_time = (time.time() - start) / 60

        # --------------------------
        # TRAIN METRICS
        # --------------------------
        results_csv = Path(results.save_dir) / "results.csv"
        df = pd.read_csv(results_csv)

        map_col = "metrics/mAP50(B)" if "metrics/mAP50(B)" in df.columns else "metrics/mAP50"
        best_row = df.loc[df[map_col].idxmax()]

        best_weights = Path(results.save_dir) / "weights/best.pt"

        # --------------------------
        # VALIDATION
        # --------------------------
        val_results = model.val(data=data_cfg)

        val_map50 = val_results.box.map50
        val_precision = val_results.box.mp
        val_recall = val_results.box.mr

        speed = val_results.speed
        total_time = sum(speed.values())
        fps = 1000 / total_time if total_time > 0 else 0

        # --------------------------
        # TEST
        # --------------------------
        test_results = model.val(data=data_cfg, split="test")

        test_map50 = test_results.box.map50
        test_precision = test_results.box.mp
        test_recall = test_results.box.mr

        # --------------------------
        # STORE
        # --------------------------
        result = {
            "seed": seed,

            "train_mAP50": best_row.get("metrics/mAP50(B)", best_row.get("metrics/mAP50")),
            "train_precision": best_row.get("metrics/precision(B)", best_row.get("metrics/precision")),
            "train_recall": best_row.get("metrics/recall(B)", best_row.get("metrics/recall")),

            "val_mAP50": val_map50,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "test_mAP50": test_map50,
            "test_precision": test_precision,
            "test_recall": test_recall,

            "fps": fps,
            "train_time_min": train_time,

            "weights": str(best_weights),
        }

        all_results.append(result)

        print("\n📊 Seed Results:")
        for k, v in result.items():
            if k != "weights":
                print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    # =====================================
    # 📊 FINAL TABLE
    # =====================================
    print("\n📊 ALL SEED RESULTS\n")

    df_results = pd.DataFrame(all_results)
    print(df_results.round(4))

    # =====================================
    # 🏆 BEST MODEL
    # =====================================
    best_exp = df_results.loc[df_results["test_mAP50"].idxmax()]

    print("\n🏆 BEST MODEL (TEST mAP50)\n")
    print(best_exp)

    # =====================================
    # 📈 AVERAGE PERFORMANCE
    # =====================================
    print("\n📈 AVERAGE PERFORMANCE\n")
    print(df_results.mean(numeric_only=True).round(4))

    # =====================================
    # 📊 PER-CLASS METRICS
    # =====================================
    print("\n📊 PER-CLASS PERFORMANCE (TEST SET)\n")

    best_model = YOLO(best_exp["weights"])

    test_results = best_model.val(data=data_cfg, split="test")

    names = test_results.names

    precision_cls = test_results.box.p
    recall_cls = test_results.box.r
    map50_cls = test_results.box.ap50

    class_metrics = []

    for i, name in names.items():
        class_metrics.append({ 
            "class": name,
            "precision": float(precision_cls[i]),
            "recall": float(recall_cls[i]),
            "mAP50": float(map50_cls[i]),
        })

    df_class = pd.DataFrame(class_metrics)
    df_class = df_class.sort_values(by="mAP50", ascending=False)

    print(df_class.round(4))

    print("\n✅ DONE\n")


# =========================================================
# ▶️ RUN
# =========================================================
run()


🚀 Starting Experiments...


========== SEED 1 ==========

New https://pypi.org/project/ultralytics/8.4.112 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.11.0 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\MY Projects\Steel Defect Detection\configs\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0

In [1]:
# =========================================================
# 📦 IMPORTS + SETUP
# =========================================================
import sys, os, random, time, gc
from pathlib import Path
import yaml
import torch
import pandas as pd

# ---------------------------------------------------------
# 🔥 PROJECT ROOT SETUP
# ---------------------------------------------------------
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] 
     if (p / "src").exists() and (p / "configs").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# 🔥 REGISTER CUSTOM MODULES
# ---------------------------------------------------------
import src
import ultralytics.nn.tasks as _tasks
import ultralytics.nn.modules as _modules

from src.custom_modules import M_C3k2, WeightedConcat, HybridSPDConv_3
from src.spd_conv import SPDConv, SPDHybrid, SPDHybrid_NO_Fuse, DKStem

for name, cls in {
    "M_C3k2": M_C3k2,
    "WeightedConcat": WeightedConcat,
    "HybridSPDConv_3": HybridSPDConv_3,
    "SPDConv": SPDConv,
    "SPDHybrid": SPDHybrid,
    "SPDHybrid_NO_Fuse": SPDHybrid_NO_Fuse,
    "DKStem": DKStem,
}.items():
    _tasks.__dict__[name] = cls
    _modules.__dict__[name] = cls

from ultralytics import YOLO


# =========================================================
# 🔁 SEED CONTROL
# =========================================================
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# 📂 LOAD CONFIG
# =========================================================
def load_config():
    cfg_path = PROJECT_ROOT / "configs" / "base.yaml"
    with open(cfg_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg["model"] = str((PROJECT_ROOT / cfg["model"]).resolve())
    cfg["experiment"]["project_dir"] = str(
        (PROJECT_ROOT / cfg["experiment"]["project_dir"]).resolve()
    )
    cfg["data"]["config"] = str(
        (PROJECT_ROOT / cfg["data"]["config"]).resolve()
    )

    return cfg


# =========================================================
# 🚀 MAIN PIPELINE
# =========================================================
def run():
    cfg = load_config()

    model_name = cfg["model"]
    training = cfg["training"]
    experiment = cfg["experiment"]
    data_cfg = cfg["data"]["config"]

    seeds = experiment["seeds"]

    all_results = []

    print("\n🚀 Starting Experiments...\n")

    # =====================================
    # 🔁 LOOP OVER SEEDS
    # =====================================
    for seed in seeds:

        print(f"\n========== SEED {seed} ==========\n")

        set_seed(seed)

        exp_name = f"{experiment['name']}_seed{seed}"

        model = YOLO(model_name)

        # --------------------------
        # TRAIN
        # --------------------------
        start = time.time()

        results = model.train(
            data=data_cfg,
            imgsz=training["imgsz"],
            batch=training["batch"],
            epochs=training["epochs"],
            optimizer=training["optimizer"],
            lr0=training["lr0"],
            workers=training["workers"],
            seed=seed,
            project=experiment["project_dir"],
            name=exp_name,
        )

        train_time = (time.time() - start) / 60

        # --------------------------
        # TRAIN METRICS
        # --------------------------
        results_csv = Path(results.save_dir) / "results.csv"
        df = pd.read_csv(results_csv)

        map_col = "metrics/mAP50(B)" if "metrics/mAP50(B)" in df.columns else "metrics/mAP50"
        best_row = df.loc[df[map_col].idxmax()]

        best_weights = Path(results.save_dir) / "weights/best.pt"

        # --------------------------
        # VALIDATION
        # --------------------------
        val_results = model.val(data=data_cfg)

        val_map50 = val_results.box.map50
        val_precision = val_results.box.mp
        val_recall = val_results.box.mr

        speed = val_results.speed
        total_time = sum(speed.values())
        fps = 1000 / total_time if total_time > 0 else 0

        # --------------------------
        # TEST
        # --------------------------
        test_results = model.val(data=data_cfg, split="test")

        test_map50 = test_results.box.map50
        test_precision = test_results.box.mp
        test_recall = test_results.box.mr

        # --------------------------
        # STORE
        # --------------------------
        result = {
            "seed": seed,

            "train_mAP50": best_row.get("metrics/mAP50(B)", best_row.get("metrics/mAP50")),
            "train_precision": best_row.get("metrics/precision(B)", best_row.get("metrics/precision")),
            "train_recall": best_row.get("metrics/recall(B)", best_row.get("metrics/recall")),

            "val_mAP50": val_map50,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "test_mAP50": test_map50,
            "test_precision": test_precision,
            "test_recall": test_recall,

            "fps": fps,
            "train_time_min": train_time,

            "weights": str(best_weights),
        }

        all_results.append(result)

        print("\n📊 Seed Results:")
        for k, v in result.items():
            if k != "weights":
                print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    # =====================================
    # 📊 FINAL TABLE
    # =====================================
    print("\n📊 ALL SEED RESULTS\n")

    df_results = pd.DataFrame(all_results)
    print(df_results.round(4))

    # =====================================
    # 🏆 BEST MODEL
    # =====================================
    best_exp = df_results.loc[df_results["test_mAP50"].idxmax()]

    print("\n🏆 BEST MODEL (TEST mAP50)\n")
    print(best_exp)

    # =====================================
    # 📈 AVERAGE PERFORMANCE
    # =====================================
    print("\n📈 AVERAGE PERFORMANCE\n")
    print(df_results.mean(numeric_only=True).round(4))

    # =====================================
    # 📊 PER-CLASS METRICS
    # =====================================
    print("\n📊 PER-CLASS PERFORMANCE (TEST SET)\n")

    best_model = YOLO(best_exp["weights"])

    test_results = best_model.val(data=data_cfg, split="test")

    names = test_results.names

    precision_cls = test_results.box.p
    recall_cls = test_results.box.r
    map50_cls = test_results.box.ap50

    class_metrics = []

    for i, name in names.items():
        class_metrics.append({ 
            "class": name,
            "precision": float(precision_cls[i]),
            "recall": float(recall_cls[i]),
            "mAP50": float(map50_cls[i]),
        })

    df_class = pd.DataFrame(class_metrics)
    df_class = df_class.sort_values(by="mAP50", ascending=False)

    print(df_class.round(4))

    print("\n✅ DONE\n")


# =========================================================
# ▶️ RUN
# =========================================================
run()


🚀 Starting Experiments...


========== SEED 1 ==========

WARNING no model scale passed. Assuming scale='n'.
Ultralytics 8.4.14  Python-3.11.0 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\MY Projects\Steel Defect Detection\configs\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=220, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\MY Projects\Steel Defect 